In [1]:
# run the rest later
%load_ext autoreload
%autoreload 2

In [2]:
import os
import subprocess
import json
from tqdm import tqdm
import logging

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import glob
from torch.utils.data import TensorDataset, DataLoader

import torch.optim as optim
from collections import OrderedDict
from multiprocessing import Pool
import matplotlib.pyplot as plt

from utilities import *
from model import *

In [3]:
print(os.environ.get('CONDA_DEFAULT_ENV'))
nvidia_smi_output = subprocess.check_output(['nvidia-smi']).decode('utf-8')
print(nvidia_smi_output)
torch.cuda.is_available()

biomedformer
Wed Aug  6 09:08:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 PCIe               Off |   00000000:17:00.0 Off |                    0 |
| N/A   59C    P0            199W /  350W |   69451MiB /  81559MiB |    100%      Default |
|                                         |                        |             Disabled |
+----------------------------------

True

In [4]:
device = torch.device('cuda:0')

with open("/home/bbc8731/BioMedFormer/data/param/parameters.json", 'r') as file:
    all_param = json.load(file)

KG_dir = all_param['files']['KG_dir']
data_dir = os.path.join(all_param['files']['processed_data'], "interpretation")
result_dir = os.path.join(all_param['files']['data_dir'], 'result/interpretation')

attr_dir = all_param['files']['node_attr']
model_dir = os.path.join(all_param['files']['data_dir'], 'saved_models/biomedformer_user')
# subnetwork_dir = "/home/bbc8731/BioMedFormer/data/result/subnetworks/"
subnetwork_dir = "//cosybio_project/faren/faren_biomedformer/subnetwork_diseases/"

model_dir_LP = "/home/bbc8731/BioMedFormer/data/saved_models/transductive_models/final_model_after_LP"
log_name_LP = "biomedformer_atr_transductive_LP_full_data_wl5_wpn30_dim300_head5_lencoder30_bs4000_lr0.0001_seed42"
checkpoint_path_LP = f"{model_dir_LP}/{log_name_LP}.pt"



In [5]:
nodes = pd.read_csv(os.path.join(KG_dir, 'nodes.csv'), sep= ',')
edge = pd.read_csv(os.path.join(KG_dir, "edges.csv")) 
diseases = nodes.loc[nodes['node_type'] == 'disease','node_index'].values
drugs = nodes.loc[nodes['node_type'] == 'drug','node_index'].values

In [6]:
# identify the edge indecies and concate with node indices
index_relations = edge[['relation', 'z_index', 'relation']].drop_duplicates()
index_relations.columns = ['node_type', 'node_index', 'node_name']
node_edge_indecices = pd.concat([nodes, index_relations])
mask_token_id = edge['z_index'].max() +1
indication_token_id = edge.loc[edge['relation'] == 'indication','z_index'].drop_duplicates().values.item()

In [7]:
# for loading the model
mask_token = edge['z_index'].max() + 1
vocab_size = mask_token + 1

relation_type = list(edge['relation'].unique())
node_types = list(nodes['node_type'].unique())
node_types_without_feat = ['biological_process',
                           'molecular_function',
                           'cellular_component',
                           'exposure',
                           'pathway',
                           'anatomy']

mask = ['mask']
entity = node_types + relation_type + mask

without_feat = {
    'node': node_types_without_feat,
    'relation': relation_type,
    'mask': mask
}

## find relation_mask_index
relation_index = edge.loc[edge['relation'].isin(relation_type), ['relation', 'z_index']].drop_duplicates()
mask_row = pd.DataFrame([['mask', mask_token]], columns=['relation', 'z_index'])
relation_mask_index = pd.concat([relation_index, mask_row], ignore_index=True)

# Generate hyperparameter combinations
param_grid = {
    'hidden_channels': [300],    
    'walk_length': [5],         
    'walks_per_node': [30],      
    'nhead': [10],               
    'N_encoder_layers': [20],     
    'batch_size': [4000],       
    'learning_rate': [0.0001]    
}

drug_pubmedbert = pd.read_csv(os.path.join(attr_dir, 'attr_drug_pubmedbert.csv'), sep= ',', index_col=0)
drug_pubmedbert = drug_pubmedbert.sort_index()
drug_pubmedbert = torch.tensor(drug_pubmedbert.values, dtype=torch.float32)  

drug_FP = pd.read_csv(os.path.join(attr_dir, 'attr_drug_fingerprint.csv'), sep= ',')
drug_FP = drug_FP.sort_index()
drug_FP = torch.tensor(drug_FP.values, dtype=torch.float32)  

gene_pubmedbert = pd.read_csv(os.path.join(attr_dir, 'attr_protein_pubmedbert.csv'), sep= ',', index_col=0)
gene_pubmedbert = gene_pubmedbert.sort_index()
gene_pubmedbert = torch.tensor(gene_pubmedbert.values, dtype=torch.float32)  

gene_esm2 = torch.load(os.path.join(attr_dir, 'attr_protein_emb_esm2.pt'))

disease_pubmedbert = pd.read_csv(os.path.join(attr_dir, 'attr_disease_pubmedbert.csv'), sep= ',', index_col = 0)
disease_pubmedbert = disease_pubmedbert.sort_index()
disease_pubmedbert = torch.tensor(disease_pubmedbert.values, dtype=torch.float32)  

phenotype_pubmedbert = pd.read_csv(os.path.join(attr_dir, 'attr_phenotype_pubmedbert.csv'), sep= ',', index_col = 0)
phenotype_pubmedbert = phenotype_pubmedbert.sort_index()
phenotype_pubmedbert = torch.tensor(phenotype_pubmedbert.values, dtype=torch.float32)  

# make a dict of all the features
feat = {
    'gene/protein': {
        'pubmedbert': gene_pubmedbert,
        'esm2': gene_esm2
    },
    'drug': {
        'pubmedbert': drug_pubmedbert,
        'FP': drug_FP
    },
    'disease': {
        'pubmedbert': disease_pubmedbert,
    },
    'effect/phenotype': {
        'pubmedbert': phenotype_pubmedbert,
    }
}

feat_rest = {}
for k, v in without_feat.items():
    if k == 'node':
        for i in v:
            N_nodes_i = (nodes['node_type'] == i).sum()
            feat_i = torch.randn(N_nodes_i, param_grid['hidden_channels'][0])
            feat_rest[f"{i}"] = {'random': feat_i}

    elif k == 'relation':
         for i in v:
             feat_i = torch.ones(1, param_grid['hidden_channels'][0])
             feat_rest[f"{i}"] = {'fixed': feat_i}   # I should put this also fixed instead of random, but now the result is based on random
    elif k == 'mask':
        feat_i = torch.ones(1,param_grid['hidden_channels'][0])
        feat_rest[f"{k}"] = {'fixed': feat_i}   

feat.update(feat_rest)


In [8]:
# load the model
tag = f"{model_dir}/biomedformer_atr_final_transductive_wl5_wpn30_dim300_head5_lencoder30_bs4000_lr0.0001_seed42_modelLastEpoch"

if len(tag) > 0:
    checkpoint_path = f"{tag}.pt"
    checkpoint_path = os.path.join(model_dir, checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    param = checkpoint['parameters']
    model = TransformerModel(
        vocab_size,
        param['hidden_channels'],
        param['nhead'],
        param['N_encoder_layers'],
        (param['walk_length']*2)-1, #walk_length_with_relation
        device=device,
        feat = feat,
        nodes = nodes,
        entity = entity,
        relation_mask_index = relation_mask_index,
        pos_emb='fixed',
    ).to(device)

    # model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    
    # Original checkpoint
    state_dict = checkpoint['model_state_dict']
    model_state = model.state_dict()

    ###### debugging
    model_params = list(model.state_dict().keys())
    checkpoint_params = list(checkpoint['model_state_dict'].keys())
    
    # What parameters exist in checkpoint but not in model
    print("Params in checkpoint but not in model:", set(checkpoint_params) - set(model_params))
    
    # What parameters exist in model but not in checkpoint
    print("Params in model but not in checkpoint:", set(model_params) - set(checkpoint_params))
    
    
    # Filter out incompatible keys
    filtered_state_dict = OrderedDict()
    for k, v in state_dict.items():                
        if k in model_state and v.shape == model_state[k].shape:
            filtered_state_dict[k] = v
        else:
            print(f"Skipping {k}: checkpoint shape {v.shape}, model shape {model_state.get(k, 'Not found').shape}")
    
    # Load filtered state dict
    model.load_state_dict(filtered_state_dict, strict=False)
    # optimizer = optim.Adam(model.parameters(), lr=param['learning_rate'])
    # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # start_epoch = checkpoint['epoch'] + 1
    # loss = checkpoint['loss']
    missing_keys = set(model_state.keys()) - set(filtered_state_dict.keys())
    if missing_keys:
        print(f"Warning: {len(missing_keys)} parameters in the model were not loaded from the checkpoint.")

    

/home/bbc8731/miniconda3/envs/biomedformer/lib/python3.10/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Params in checkpoint but not in model: set()
Params in model but not in checkpoint: set()


In [9]:
# load the model
tag_LP = f"{model_dir_LP}/{log_name_LP}"

checkpoint_path_LP = f"{tag_LP}.pt"
checkpoint_path_LP = os.path.join(model_dir_LP, checkpoint_path_LP)
checkpoint_LP = torch.load(checkpoint_path_LP, map_location=device)

param_LP = checkpoint_LP['parameters']

classifier = Classifier(param_LP['hidden_channels_encoder'], param_LP['d_classifier'], device)
state_dict_LP = checkpoint_LP['classifier_state_dict']

# Load state dict
classifier.load_state_dict(state_dict_LP)


<All keys matched successfully>

In [10]:
### generate walk with disaese and indication relation to give to the model for finding top_k drugs
sentence = torch.ones(len(diseases), param['walk_length'] * 2 - 1) * mask_token_id
sentence[:, 0] = torch.tensor(diseases, dtype=torch.long)
sentence[:, 1] = indication_token_id
sentence = sentence.to(torch.long)


In [ ]:
# save the conversion of the disease_id which is range(len(disease_nodes))  and names of the diseases to send to Simon for UI
disease_nodes = nodes.loc[nodes['node_type']== 'disease',['node_name']]
disease_nodes = disease_nodes.copy()
disease_nodes['disease_id'] = range(len(disease_nodes))
disease_nodes = disease_nodes.loc[:, ['disease_id', 'node_name']]
disease_nodes
disease_nodes.to_csv('/home/bbc8731/BioMedFormer/disease_nodes.csv', index=False)


In [11]:
##### model1
# find output of the model to find top_k drugs
dataset = TensorDataset(sentence)
dataloader = DataLoader(dataset, batch_size=512, shuffle=False)
N_top_drugs = 100

model.eval()
drugs = torch.tensor(drugs).to(device)
top_drugs_all = []
top_probs_drugs_all = []
for batch in tqdm(dataloader):
    input_batch = batch[0].to(device)
    with torch.no_grad():
        output = model(input_batch)  # shape: (batch_size, seq_len, vocab_size)
        logits = output[:, 2, drugs] # shape: (batch_size, num_drugs)        
        probs = F.softmax(logits, dim=1)  # apply softmax across the drugs

        top_probs, top_drugs = torch.topk(probs, N_top_drugs, dim=1)  # get top-k based on probabilities
        # top_logits, top_drugs = torch.topk(output[:, 2, drugs], N_top_drugs, dim=1)  # (batch_size, 100)
        top_drugs = drugs[top_drugs]  # map indices back to drug IDs
        top_drugs_all.append(top_drugs.cpu())  
        top_probs_drugs_all.append(top_probs.cpu())  # move to CPU if needed

# Concatenate all results
top_drugs_all = torch.cat(top_drugs_all, dim=0)
top_probs_drugs_all = torch.cat(top_probs_drugs_all, dim=0)

print(top_drugs_all.shape)  # should be (num_sentences, 100)


100%|███████████████████████████████████████████| 34/34 [00:13<00:00,  2.54it/s]

torch.Size([17080, 100])


In [12]:
##### model3

# # find output of the model to find top_k drugs
# dataset = TensorDataset(sentence)
# dataloader = DataLoader(dataset, batch_size=512, shuffle=False)
# N_top_drugs = 100

# model.eval()
# drugs = torch.tensor(drugs).to(device)
# top_drugs_all = []
# top_probs_drugs_all = []
# for batch in tqdm(dataloader):
#     input_batch = batch[0].to(device)
#     with torch.no_grad():
#         output = model(input_batch)  # shape: (batch_size, seq_len, vocab_size)
#         logits = output[:, 2, :] # shape: (batch_size, num_drugs)

#         mask_logits = torch.full_like(logits, float('-inf'))
#         mask_logits[:, drugs] = logits[:, drugs]
#         logits = mask_logits
        
#         probs = F.softmax(logits, dim=1)  # apply softmax across the drugs

#         top_probs, top_drugs = torch.topk(probs, N_top_drugs, dim=1)  # get top-k based on probabilities
        
#         # top_logits, top_drugs = torch.topk(output[:, 2, drugs], N_top_drugs, dim=1)  # (batch_size, 100)
#         # top_drugs = drugs[top_drugs]  # map indices back to drug IDs
#         top_drugs_all.append(top_drugs.cpu())  
#         top_probs_drugs_all.append(top_probs.cpu())  # move to CPU if needed

# # Concatenate all results
# top_drugs_all = torch.cat(top_drugs_all, dim=0)
# top_probs_drugs_all = torch.cat(top_probs_drugs_all, dim=0)

# print(top_drugs_all.shape)  # should be (num_sentences, 100)


In [13]:
# plt.hist(np.array(top_probs_drugs_all[10,:]), 100);

In [14]:
# top_probs_drugs_all
###########
# plt.hist(top_probs_drugs_all.flatten(), bins=1000, edgecolor='black')
# plt.xlabel("Probability")
# plt.ylabel("Frequency")
# plt.title("Distribution of Top Probabilities for Drugs")
# plt.grid(True)
# plt.xlim([0, 0.2])
# plt.show()

In [15]:
def get_logits(walk, position, model):
    
    """
    For a given walk and position, outputs logits of all tokens for on that position
    """
    model.eval()
    outputs_list = []
    with torch.no_grad():
        output = model(walk)
        outputs_list.append(output)
        
    stacked_outputs = torch.stack(outputs_list)  # (rep, 1, seq_len, vocab_size)
    logits_at_mask = stacked_outputs[:, 0, position, :][0]  # (rep, vocab_size)
    return logits_at_mask

def get_top_k_tokens_from_logits(logits, k, node_edge_indecices):
    topk_logits, topk_indices = torch.topk(logits, k=k)
    topk_names = node_edge_indecices.set_index("node_index").loc[topk_indices.cpu().numpy()]["node_name"]
    return topk_names, topk_logits
    
def get_token_from_walk(walk, position, model, node_edge_indecices, k=5, T=1.0):
    logits = get_logits(walk, position, model)
    topk_tokens, topk_logits = get_top_k_tokens_from_logits(logits, k, node_edge_indecices)
    # print(topk_tokens)
    probs = F.softmax(topk_logits, dim=-1)

    # Apply temprature method
    scaled_logits = topk_logits / T
    probs_scaled = F.softmax(scaled_logits, dim=-1)

    # print(probs_scaled)
    sampled_index = torch.multinomial(probs_scaled, num_samples=1).item()
    token = topk_tokens.index[sampled_index]
    # print(token)
    prob_scaled = probs_scaled[sampled_index].item()
    prob = probs[sampled_index].item()

    # print(f"prob_scaled:{prob_scaled}"), print(f"prob:{prob}")
    return token, prob_scaled, prob

def metapath_generation(walk_input, n_metapath, mask_token, model, node_edge_indecices):
    """
    Given a walk_input, this function sequqntally performs next 'mask' token prediction.
    """
    k = 5
    L = len(walk_input[0])

    all_metapath = []
    all_probs_metapath = []
    all_probs_scaled_metapath = []
    for rep in range(n_metapath):
        walk = walk_input.clone()
        probs = torch.ones_like(walk).to(torch.double)
        probs_scaled = torch.ones_like(walk).to(torch.double)

        for pos in range(L):
            if walk[0, pos].item() == mask_token:
                topk_tokens, topk_probs_scaled, topk_probs = get_token_from_walk(walk, pos, model, node_edge_indecices, k=5, T=10)
                walk[0, pos] = topk_tokens 
                probs_scaled[0, pos] = topk_probs_scaled
                probs[0, pos] = topk_probs

        all_metapath.append(walk)
        all_probs_scaled_metapath.append(probs_scaled)
        all_probs_metapath.append(probs)

    return all_metapath, all_probs_scaled_metapath, all_probs_metapath 

In [16]:
# np.where(diseases == 28208)[0]

In [ ]:
model.eval()
subnetwork_dir = "//cosybio_project/faren/faren_biomedformer/subnetwork_diseases/"
log_file_path = os.path.join("/home/bbc8731/BioMedFormer/data/result/subnetworks/subnetwork.log")

logging.basicConfig(
    filename=log_file_path,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

n_metapath = 10
thr = 0.7

index_name_map = dict(zip(node_edge_indecices["node_index"], node_edge_indecices["node_name"]))
index_type_map = dict(zip(node_edge_indecices["node_index"], node_edge_indecices["node_type"]))
indication_edges = set(edge.loc[(edge['relation'] == 'indication') & (edge['node_types'] == 'disease|drug'), 'xy'])

metapath_dict = {}
all_triplets = []  
for i in tqdm(range(top_drugs_all.shape[0]), desc="Diseases"):
    logging.info(f"Processing pheno index: {i}")
    all_triplets = []
    metapath_dict = {} 

    for j in tqdm(range(top_drugs_all.shape[1])):
        disease_index = diseases[i]
        drug_index = top_drugs_all[i, j].item()
        is_existing_disease_drug = f'{disease_index}|{drug_index}' in indication_edges
        
        sen = torch.ones(param['walk_length'] * 2 - 1) * mask_token_id
        sen[0] = torch.tensor(disease_index, dtype=torch.long)
        sen[1] = indication_token_id
        sen[2] = torch.tensor(drug_index, dtype=torch.long)
        sen = sen.to(torch.long).to(device)
        
        sen = sen.unsqueeze(0)  
        emb = model.get_embeddings(sen, select = 'transformer')
        y_hat = classifier(emb)
        prob_LP = F.sigmoid(y_hat).item()
        
        if prob_LP < thr:
            continue
        
        walk_input1 = [disease_index, indication_token_id, drug_index] + [mask_token_id] * 6
        walk_input2 = [drug_index, indication_token_id, disease_index] + [mask_token_id] * 6

        walk_input1 = torch.tensor([walk_input1], device=device)
        walk_input2 = torch.tensor([walk_input2], device=device)

        metapath1, probs1_scaled, probs1 = metapath_generation(walk_input1, n_metapath, mask_token_id, model, node_edge_indecices)
        metapath2, probs2_scaled, probs2 = metapath_generation(walk_input2, n_metapath, mask_token_id, model, node_edge_indecices)

        metapath = torch.cat(metapath1 + metapath2, dim=0).detach().cpu().numpy()
        probs = torch.cat(probs1 + probs2, dim=0).detach().cpu().numpy()
        probs_scaled = torch.cat(probs1_scaled + probs2_scaled, dim=0).detach().cpu().numpy()

        triplets = []
        prob_triplets = []
        prob_scaled_triplets = []
        for row, prob_scaled, prob in zip(metapath, probs_scaled, probs):
            # Extract triplets with sliding window of size 3 and stride 2
            for k in range(0, len(row) - 2, 2):
                triplets.append(row[k:k+3].tolist())
                prob_triplets.append(prob[k:k+3].tolist())
                prob_scaled_triplets.append(prob_scaled[k:k+3].tolist())
                
        triplet_df = pd.DataFrame(triplets, columns=["head_index", "relation_index", "tail_index"])
        prob_triplets_df = pd.DataFrame(prob_triplets, columns=["head_prob", "relation_prob", "tail_prob"])
        prob_scaled_triplets_df = pd.DataFrame(prob_scaled_triplets, columns=["head_prob_scaled", "relation_prob_scaled", "tail_prob_scaled"])

        triplet_df = pd.concat([triplet_df, prob_triplets_df, prob_scaled_triplets_df], axis=1)
        
        triplet_df["head_type"] = triplet_df["head_index"].map(index_type_map)
        triplet_df["relation_type"] = triplet_df["relation_index"].map(index_type_map)
        triplet_df["tail_type"] = triplet_df["tail_index"].map(index_type_map)
        triplet_df["head_name"] = triplet_df["head_index"].map(index_name_map)
        triplet_df["tail_name"] = triplet_df["tail_index"].map(index_name_map)
        triplet_df["pair_id"] = f"{disease_index}|{drug_index}"
        triplet_df["drug_disease_score"] = prob_LP
        triplet_df["drug_disease_in_KG"] = is_existing_disease_drug
        
        all_triplets.append(triplet_df)

        key = f"{disease_index}_{drug_index}"
        metapath_dict[key] = {
            'path': metapath.tolist(),
            'probs': probs.tolist()
        }

    final_df = pd.concat(all_triplets, ignore_index=True)

    # Save file for each disease
    final_csv_path = os.path.join(subnetwork_dir, f"metapath_triplets_disease_{i}.csv")
    final_df.to_csv(final_csv_path, index=False)

    # Save metapaths into one JSON for each disease
    final_json_path = os.path.join(subnetwork_dir, f"metapaths_disease_{i}.json")  
    with open(final_json_path, "w") as f:
        json.dump(metapath_dict, f)
    
    print(f"Saved combined CSV to: {final_csv_path}")
    print(f"Saved all metapaths JSON to: {final_json_path}")

 10%|████                                    | 10/100 [16:25<2:19:42, 93.14s/it]

In [1]:
i

NameError: name 'i' is not defined

In [18]:
# y = np.array(y)
# probs = np.array(probs)
# plt.boxplot([probs[y], probs[~y]]);


# plt.hist(probs,100);
# (np.array(probs)>0.6).sum()

In [ ]:
# metapath_dict = {}
# n_metapath = 10
# thr = 0.8

# for i in tqdm(range(top_drugs_all.shape[0])):
#     for j in tqdm(range(top_drugs_all.shape[1])):
#         print(top_probs_drugs_all[i,j])
        
#         if top_probs_drugs_all[i,j]<thr:
#             continue

#         disease_index = diseases[i]
#         drug_index = top_drugs_all[i, j].item()
        
#         walk_input1 = [disease_index, indication_token_id, drug_index]+ [mask_token_id]*6
#         walk_input1 = torch.tensor([walk_input1],  device=device)
#         walk_input2 = [drug_index, indication_token_id, disease_index]+ [mask_token_id]*6
#         walk_input2 = torch.tensor([walk_input2],  device=device)
        
#         metapath1, probs1_scaled, probs1 = metapath_generation(walk_input1, n_metapath, mask_token_id, model, node_edge_indecices)
#         metapath2, probs2_scaled, probs2 = metapath_generation(walk_input2, n_metapath, mask_token_id, model, node_edge_indecices)
                        
#         metapath = torch.cat(metapath1 + metapath2, dim = 0)
#         metapath = metapath.detach().cpu().numpy()
#         probs = torch.cat(probs1 + probs2, dim = 0)
#         probs_scaled = torch.cat(probs1_scaled + probs2_scaled, dim = 0)

#         probs = probs.detach().cpu().numpy()
#         probs_scaled = probs_scaled.detach().cpu().numpy()

#         triplets = []
#         prob_triplets = []
#         prob_scaled_triplets = []

#         for row, prob_scaled, prob in tqdm(zip(metapath, probs_scaled, probs)):
#             # Extract triplets with sliding window of size 3 and stride 2
#             for k in range(0, len(row) - 2, 2):
#                 triplet = row[k:k+3].tolist()
#                 triplets.append(triplet)
#                 prob_triplets.append(prob[k:k+3].tolist())
#                 prob_scaled_triplets.append(prob_scaled[k:k+3].tolist())
                
#         triplet_df = pd.DataFrame(triplets, columns=["head_index", "relation_index", "tail_index"])
#         prob_triplets_df = pd.DataFrame(prob_triplets, columns=["head_prob", "relation_prob", "tail_prob"])
#         prob_scaled_triplets_df = pd.DataFrame(prob_scaled_triplets, columns=["head_prob_scaled", "relation_prob_scaled", "tail_prob_scaled"])

#         triplet_df = pd.concat([triplet_df, prob_triplets_df, prob_scaled_triplets_df], axis=1)
#         index_name_map = dict(zip(node_edge_indecices["node_index"], node_edge_indecices["node_name"]))
#         index_type_map = dict(zip(node_edge_indecices["node_index"], node_edge_indecices["node_type"]))
        
#         triplet_df["head_type"] = triplet_df["head_index"].map(index_type_map)
#         triplet_df["relation_type"] = triplet_df["relation_index"].map(index_type_map)
#         triplet_df["tail_type"] = triplet_df["tail_index"].map(index_type_map)
#         triplet_df["head_name"] = triplet_df["head_index"].map(index_name_map)
#         triplet_df["tail_name"] = triplet_df["tail_index"].map(index_name_map)
#         # triplet_df.to_csv(os.path.join(subnetwork_dir, f"edge_list_{disease_index}_{drug_index}.csv"), index=None)
        
        
#         key = f"{disease_index}_{drug_index}"
#         metapath_dict[key] = {
#             'path': metapath.tolist(),
#             'probs': probs.tolist()
#         }        
        
#         # with open(os.path.join(subnetwork_dir, f"path_set_{disease_index}_{drug_index}.json"), 'w') as file:
#         #     json.dump(metapath_dict, file)




In [15]:
triplet_df['xyz'] = [f"{row['head_index']}|{row['relation_index']}|{row['tail_index']}" for _,row in triplet_df.iterrows()]
triplet_df.drop_duplicates('xyz').shape


(59, 16)

In [25]:
def get_triplet_set(df, pair_id):
    group = df[df["pair_id"] == pair_id]
    triplets = set(zip(group["head_index"], group["relation_index"], group["tail_index"]))
    return triplets

# Define your two pair_ids
pair1 = "27158|17215"
pair2 = "99869|20180"

# Get triplet sets
triplets_1 = get_triplet_set(df, pair1)
triplets_2 = get_triplet_set(df, pair2)

# Compute overlap
intersection = triplets_1 & triplets_2  # set intersection
union = triplets_1 | triplets_2         # optional: union for Jaccard

# Print stats
print(f"Overlap between {pair1} and {pair2}: {len(intersection)} triplets")
print(f"Triplets in {pair1}: {len(triplets_1)}")
print(f"Triplets in {pair2}: {len(triplets_2)}")
print(f"Jaccard similarity: {len(intersection) / len(union):.3f}")
print("Overlapping triplets:")


for triplet in intersection:
    print(triplet)


Overlap between 27158|17215 and 99869|20180: 0 triplets
Triplets in 27158|17215: 56
Triplets in 99869|20180: 57
Jaccard similarity: 0.000
Overlapping triplets:


In [11]:
# final_csv_path = os.path.join(subnetwork_dir, "all_metapath_triplets.csv")
# final_df = pd.read_csv(final_csv_path)
# final_df

,head_index,relation_index,tail_index,head_prob,relation_prob,tail_prob,head_prob_scaled,relation_prob_scaled,tail_prob_scaled,head_type,relation_type,tail_type,head_name,tail_name,pair_id
0,27158,129389,17215,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,disease,indication,drug,osteogenesis imperfecta,Salmon calcitonin,27158|17215
1,17215,129387,23901,1.000000,0.142662,0.197931,1.000000,0.217048,0.199865,drug,drug_effect,effect/phenotype,Salmon calcitonin,Arthralgia,27158|17215
2,23901,129391,32993,0.197931,0.208781,0.181591,0.199865,0.246125,0.198652,effect/phenotype,disease_phenotype_positive,disease,Arthralgia,adult-onset Still disease,27158|17215
3,32993,129394,38171,0.181591,0.033227,0.046351,0.198652,0.205391,0.184004,disease,disease_disease,disease,adult-onset Still disease,systemic diseases with anterior uveitis,27158|17215
4,27158,129389,17215,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,disease,indication,drug,osteogenesis imperfecta,Salmon calcitonin,27158|17215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57515,9714,129381,27158,0.469609,0.213852,0.139237,0.222051,0.207200,0.194218,gene/protein,disease_protein,disease,SPARC,osteogenesis imperfecta,99892|17215
57516,17215,129389,99892,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,drug,indication,disease,Salmon calcitonin,LRP5-related primary osteoporosis,99892|17215
57517,99892,129389,17735,1.000000,0.060096,0.013166,1.000000,0.182403,0.183776,disease,indication,drug,LRP5-related primary osteoporosis,Incadronic acid,99892|17215
57518,17735,129387,23901,0.013166,0.006575,0.200339,0.183776,0.179778,0.200463,drug,drug_effect,effect/phenotype,Incadronic acid,Arthralgia,99892|17215


In [33]:
disease_index, drug_index = 39820, 14158

sen = torch.ones(param['walk_length'] * 2 - 1) * mask_token_id
sen[0] = torch.tensor(disease_index, dtype=torch.long)
sen[1] = indication_token_id
sen[2] = torch.tensor(drug_index, dtype=torch.long)
sen = sen.to(torch.long).to(device)

sen = sen.unsqueeze(0)  
emb = model.get_embeddings(sen, select = 'transformer')
y_hat = classifier(emb)
prob = F.sigmoid(y_hat).item()
y_hat.item()

7.59995174407959

In [113]:
indication_token_id

129389